# Donor Inactivity Risk Model — Master CRISP-DM Pipeline

**Self-contained, stakeholder-ready notebook.**  
Run all cells top-to-bottom for a complete end-to-end reproduction of the pipeline.  
Intermediate artifacts are loaded from `data/processed/` and `artifacts/` when available; if missing, they are regenerated automatically.

---

## Executive Summary

This pipeline scores every active donor on a **0–1 risk probability** representing their likelihood of going inactive within the next 90 days.  
Board members and admins review a weekly prioritized list (sorted highest score first) and conduct personal outreach calls to high-risk donors before the relationship lapses.

| | |
|---|---|
| **Problem type** | Binary classification |
| **Target** | `at_risk` — 1 if donor went inactive, 0 otherwise |
| **Primary metric** | Recall ≥ 0.70 (minimize missed at-risk donors) |
| **Secondary metric** | ROC AUC ≥ 0.75 |
| **Model** | Random Forest (selected over Logistic Regression by +0.10 CV recall) |
| **Decision** | ⚠ Conditional Go — Recall threshold met; ROC AUC marginally below threshold (small dataset caveat applies) |
| **Prediction sink** | `donor_risk_scores` table in Supabase |
| **Retraining cadence** | Quarterly, or when donor count grows by 50% |

---

## Pipeline Phases
1. [Business Understanding](#phase1)
2. [Data Understanding](#phase2)
3. [Data Preparation](#phase3)
4. [Modeling](#phase4)
5. [Evaluation](#phase5)

---
## Setup — Imports & Data Loading
<a id="setup"></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import json
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Allow imports from src/
NOTEBOOK_DIR = Path('..').resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from src.config import (
    SEED, TEST_SIZE, AT_RISK_DAYS, TARGET,
    ARTIFACTS_MODELS, ARTIFACTS_RUNS,
)
from src.features import build_donation_features, add_label
from src.modeling import build_preprocessor
from src.metrics import report_classification

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay,
    average_precision_score, PrecisionRecallDisplay,
    recall_score, precision_score, f1_score,
)

sns.set_theme(style='whitegrid', palette='muted')

# ── Data loading helper ────────────────────────────────────────────────────
RAW_DIR      = NOTEBOOK_DIR / 'data' / 'raw'
PROCESSED_DIR = NOTEBOOK_DIR / 'data' / 'processed'
DATASETS_DIR  = NOTEBOOK_DIR.parent.parent / 'datasets'  # repo-level datasets/

def find_csv(filename):
    """Search raw/, then repo datasets/ for a CSV file."""
    for candidate in [
        RAW_DIR / filename,
        DATASETS_DIR / filename,
    ]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{filename} not found in data/raw/ or datasets/. "
        "Place the file in pipeline/inactive_donor_pred/data/raw/ and re-run."
    )

supporters = pd.read_csv(find_csv('supporters.csv'), parse_dates=['created_at', 'first_donation_date'])
donations  = pd.read_csv(find_csv('donations.csv'),  parse_dates=['donation_date'])

print(f'supporters : {supporters.shape[0]} rows × {supporters.shape[1]} columns')
print(f'donations  : {donations.shape[0]} rows × {donations.shape[1]} columns')
print(f'AT_RISK_DAYS window : {AT_RISK_DAYS} days')

---
## Phase 1: Business Understanding
<a id="phase1"></a>
**CRISP-DM Purpose:** Define the business problem, objectives, and success criteria before any data work begins.

### 1.1 Problem Statement

| | |
|---|---|
| **Business Question** | Which active donors are most likely to go inactive within the next 90 days, so that board members and admins can prioritize personal outreach before the relationship lapses? |
| **Target Variable** | `at_risk` (engineered) |
| **Problem Type** | Binary Classification |
| **Positive Class** | `1 = at risk of going inactive` |
| **Prediction Horizon** | 90 days |
| **Output** | Risk score (0–1 probability) per active donor, written to `donor_risk_scores` in Supabase |
| **Consumer** | Board members and admins — review a prioritized list and conduct personal outreach calls |

### 1.2 Feasibility Assessment

| Criterion | Assessment |
|---|---|
| **Practical Impact** | Retaining a lapsing donor is significantly cheaper than acquiring a new one. Early identification enables personal outreach before the relationship is lost. |
| **Data Availability** | Two tables: `supporters` (demographics, acquisition channel, status) and `donations` (history, type, recency, amount). Both in Supabase, exportable as CSV. |
| **Analytical Feasibility** | Sufficient features exist to build behavioral profiles (recency, frequency, monetary value, donor type, channel). Dataset is small (~60 donors) — model must be regularized. |
| **Label Feasibility** | `at_risk` label is engineered from donation recency (no donation in 90 days). Reasonable and defensible proxy for inactivity risk. |

### 1.3 Success Criteria

| | |
|---|---|
| **Primary metric** | Recall (at-risk class) — minimize missed at-risk donors |
| **Secondary metric** | ROC AUC — overall discrimination ability |
| **Minimum acceptable Recall** | ≥ 0.70 |
| **Minimum acceptable ROC AUC** | ≥ 0.75 |

> **Why Recall?** The cost of a false negative (missing an at-risk donor, losing them silently) far outweighs the cost of a false positive (an unnecessary personal call to a loyal donor).

### 1.4 Error Cost Analysis

| Error Type | Scenario | Cost |
|---|---|---|
| **False Negative** | Model predicts safe → no outreach → donor goes inactive | High — lost donor relationship |
| **False Positive** | Model flags loyal donor → unnecessary call | Low — a personal call rarely harms a relationship |

### 1.5 Stakeholder Impact

| | |
|---|---|
| **Primary users** | Board members and admins |
| **Workflow** | Weekly: run inference → review prioritized risk list → assign personal outreach calls |
| **Decisions automated** | None — model surfaces risk, humans decide who to call |
| **Prediction sink** | `donor_risk_scores` table in Supabase, keyed by `supporter_id` |

In [ ]:
# Build features + label, then compute majority-class baseline
df = build_donation_features(supporters, donations)
df = add_label(df, at_risk_days=AT_RISK_DAYS)

active = df[df['status'] == 'Active'].copy()
label_counts = active[TARGET].value_counts()
majority_baseline = label_counts.max() / label_counts.sum()

print(f'Active donors  : {len(active)}')
print(f'All donors     : {len(df)}')
print(f'\nat_risk label distribution (among active donors):')
print(label_counts.rename({0: 'Not at risk (0)', 1: 'At risk (1)'}))
print(f'\nMajority-class baseline accuracy : {majority_baseline:.1%}')

### Phase 1 Conclusion

Problem is well-defined and feasible. We are building a binary classifier to score active donors by their probability of going inactive within 90 days. The key risk is dataset size (~60 donors) — results are directionally useful now but will improve as more donors are added.

---
## Phase 2: Data Understanding
<a id="phase2"></a>
**CRISP-DM Purpose:** Become familiar with the data's structure, quality, variables, and relationships before any transformation.

In [ ]:
print('=== SUPPORTERS — sample rows ===')
display(supporters.head(3))
print('\n=== DONATIONS — sample rows ===')
display(donations.head(3))

In [ ]:
def missingness_report(df, name):
    desc = df.describe(include='all').T
    desc['missing']     = df.isnull().sum()
    desc['missing_pct'] = (df.isnull().mean() * 100).round(1)
    desc['nunique']     = df.nunique()
    desc['dtype']       = df.dtypes
    cols = ['dtype', 'count', 'missing', 'missing_pct', 'nunique']
    print(f'\n=== {name} — Missingness Report ===')
    display(desc[cols])

missingness_report(supporters, 'SUPPORTERS')
missingness_report(donations,  'DONATIONS')

### 2.1 Data Quality Notes

| Column | Null Behaviour | Classification |
|---|---|---|
| `supporters.organization_name` | Null for individuals | **By design** |
| `supporters.first_name` / `last_name` | Null for PartnerOrganization rows | **By design** |
| `supporters.first_donation_date` | 1 null observed | **Acceptable** |
| `donations.amount` | Null for non-monetary types | **By design** — use `estimated_value` instead |
| `donations.currency_code` | Null for non-monetary | **By design** |
| `donations.campaign_name` | Null if not campaign-driven | **Expected** |
| `donations.referral_post_id` | Null if not social-referred | **Expected** |

> **Key decision:** Use `estimated_value` (not `amount`) for all monetary aggregations — it is populated for all donation types.

In [ ]:
# Target distribution
label_counts_all = df[TARGET].value_counts()
ratio = label_counts_all.max() / label_counts_all.min()

print(f'at_risk distribution (all 60 donors — training scope):')
print(label_counts_all.rename({0: 'Not at risk (0)', 1: 'At risk (1)'}))
print(f'\nImbalance ratio : {ratio:.1f}:1')
if ratio >= 4:
    print('⚠ Imbalance ≥ 4:1 — class_weight="balanced" required.')
else:
    print('✓ Imbalance < 4:1 — class_weight="balanced" applied as precaution.')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
supporters['status'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'salmon'], rot=0)
axes[0].set_title('Supporter Status (all 60)')
axes[0].set_ylabel('Count')

label_counts_all.rename({0: 'Not at risk', 1: 'At risk'}).plot(
    kind='bar', ax=axes[1], color=['steelblue', 'salmon'], rot=0)
axes[1].set_title(f'at_risk Label (all donors, {AT_RISK_DAYS}d window)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature distributions
cat_cols = ['supporter_type', 'relationship_type', 'region', 'acquisition_channel']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flat, cat_cols):
    supporters[col].value_counts().plot(kind='bar', ax=ax, rot=30, color='steelblue')
    ax.set_title(col)
    ax.set_ylabel('Count')
plt.suptitle('Supporter Categorical Feature Distributions', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# at_risk rate by categorical features (active donors)
cat_features = ['supporter_type', 'relationship_type', 'acquisition_channel', 'region']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, cat_features):
    risk_by_cat = active.groupby(col)[TARGET].mean().sort_values(ascending=False)
    risk_by_cat.plot(kind='bar', ax=ax, color='salmon', rot=30)
    ax.set_title(f'at_risk rate by {col}')
    ax.set_ylabel('Proportion at risk')
    ax.set_ylim(0, 1)
    ax.axhline(active[TARGET].mean(), color='steelblue', linestyle='--', label='Overall avg')
    ax.legend(fontsize=8)
plt.suptitle('at_risk Rate by Categorical Features (active donors)', y=1.01)
plt.tight_layout()
plt.show()

print('at_risk rate by has_recurring (active donors):')
print(active.groupby('has_recurring')[TARGET].mean().rename({0: 'Non-recurring', 1: 'Recurring'}))

print('\nat_risk rate by supporter_type:')
print(
    active.groupby('supporter_type')[TARGET]
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
    .round(2)
)

In [ ]:
# Temporal scope
print(f'Supporters created_at range : {supporters["created_at"].min().date()} → {supporters["created_at"].max().date()}')
print(f'Donations date range        : {donations["donation_date"].min().date()} → {donations["donation_date"].max().date()}')

donations['month'] = donations['donation_date'].dt.to_period('M')
monthly = donations.groupby('month').size()

fig, ax = plt.subplots(figsize=(13, 4))
monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Donation Volume by Month')
ax.set_ylabel('Number of Donations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print('No seasonal pattern expected (user-confirmed).')

### Phase 2 Conclusion

Data is well-understood and ready for preparation. All nulls are by design or low-impact. Most predictive features: `has_recurring`, `acquisition_channel`, `supporter_type`, `total_donations`, `avg_days_between_donations`. No seasonality in donation patterns.

---
## Phase 3: Data Preparation
<a id="phase3"></a>
**CRISP-DM Purpose:** Transform raw data into modeling-ready form. All cleaning, feature engineering, and splitting decisions are made and documented here.

In [ ]:
# Training scope: ALL 60 donors (active + inactive)
# Inactive donors ARE the ground truth positive class — they've already lapsed.
df_model = df.copy().reset_index(drop=True)

print(f'Training scope : {len(df_model)} donors (active + inactive)')
print(f'at_risk distribution:')
print(df_model[TARGET].value_counts().rename({0: 'Active (not at risk)', 1: 'Inactive (at risk)'}))

In [ ]:
EXCLUDE = {
    'supporter_id':             'Identifier — no predictive signal',
    'display_name':             'PII — free-text name',
    'organization_name':        'PII — sparse, mostly null for individuals',
    'first_name':               'PII',
    'last_name':                'PII',
    'email':                    'PII',
    'phone':                    'PII',
    'status':                   '⚠ LEAKAGE — directly encodes the label',
    'created_at':               'Encoded as supporter_tenure_days',
    'first_donation_date':      'Encoded inside build_donation_features',
    'last_donation_date':       'Raw date — already encoded as days_since_last_donation',
    'first_donation_date_d':    'Raw date — already encoded as donor_tenure_days',
    'days_since_last_donation': '⚠ LEAKAGE — inactive donors have high recency by definition',
    TARGET:                     'Target column',
}

print('Excluded columns and reasons:')
for col, reason in EXCLUDE.items():
    print(f'  {col:<30} → {reason}')

feature_cols     = [c for c in df_model.columns if c not in EXCLUDE]
numeric_cols     = df_model[feature_cols].select_dtypes(include='number').columns.tolist()
categorical_cols = df_model[feature_cols].select_dtypes(exclude='number').columns.tolist()

print(f'\nFeature columns ({len(feature_cols)}) : {feature_cols}')
print(f'Numeric     ({len(numeric_cols)}) : {numeric_cols}')
print(f'Categorical  ({len(categorical_cols)}) : {categorical_cols}')

In [ ]:
# ── Load from processed/ if available; otherwise split and save ───────────
if (PROCESSED_DIR / 'train.csv').exists() and (PROCESSED_DIR / 'test.csv').exists():
    train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
    test_df  = pd.read_csv(PROCESSED_DIR / 'test.csv')
    X_train, y_train = train_df[feature_cols], train_df[TARGET]
    X_test,  y_test  = test_df[feature_cols],  test_df[TARGET]
    print('Loaded splits from data/processed/')
else:
    from sklearn.model_selection import train_test_split
    X = df_model[feature_cols]
    y = df_model[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    X_train.assign(**{TARGET: y_train}).to_csv(PROCESSED_DIR / 'train.csv', index=False)
    X_test.assign(**{TARGET: y_test}).to_csv(PROCESSED_DIR / 'test.csv', index=False)
    with open(PROCESSED_DIR / 'feature_cols.json', 'w') as f:
        json.dump({'feature_cols': feature_cols, 'numeric_cols': numeric_cols,
                   'categorical_cols': categorical_cols}, f, indent=2)
    print('Splits generated and saved to data/processed/')

print(f'Train : {len(X_train)} rows  |  Test : {len(X_test)} rows (frozen)')
print(f'Train at_risk rate : {y_train.mean():.1%}')
print(f'Test  at_risk rate : {y_test.mean():.1%}')

In [ ]:
# Preprocessing pipeline (never fit on full dataset before splitting)
# | Numeric     | Median imputation + StandardScaler                    |
# | Categorical | Most-frequent imputation + OneHotEncoder               |
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

# Dry-run to confirm output shape
X_transformed = preprocessor.fit_transform(X_train)
print(f'Preprocessor output shape : {X_transformed.shape}')
print(f'  {len(numeric_cols)} numeric + {X_transformed.shape[1] - len(numeric_cols)} OHE-expanded categorical columns')

# Imbalance check
counts = y_train.value_counts()
ratio  = counts.max() / counts.min()
print(f'\nTraining class counts:\n{counts.rename({0: "Not at risk", 1: "At risk"})}')
print(f'Imbalance ratio : {ratio:.1f}:1')
if ratio >= 4:
    print('⚠ Imbalance ≥ 4:1 → class_weight="balanced" required in Phase 4.')
else:
    print('✓ Imbalance < 4:1 — class_weight="balanced" applied as precaution.')

### Phase 3 Conclusion

| Decision | Choice | Reason |
|---|---|---|
| Training scope | All 60 donors | Inactive donors are ground truth positives |
| `status` excluded | Yes | Directly encodes the label — leakage |
| `days_since_last_donation` excluded | Yes | Inactive donors have high recency by definition — leakage |
| Numeric imputation | Median | Robust to right-skewed donation amounts |
| Encoding | OneHotEncoder (handle_unknown='ignore') | Handles unseen categories at inference |
| Split | 80/20 stratified | Preserves class ratio; test set frozen |

---
## Phase 4: Modeling
<a id="phase4"></a>
**CRISP-DM Purpose:** Train and compare candidate models using cross-validation on the training set only. The test set is **not touched here**.

### 4.1 Candidate Models

| Model | Rationale | Resource profile |
|---|---|---|
| **Logistic Regression** | Coefficients are directly interpretable — each feature has a signed weight showing direction and magnitude of risk. | Very lightweight |
| **Random Forest (100 trees)** | Non-linear benchmark. Feature importances provide a secondary explainability signal. | Moderate — kept small for low-resource deployment |

In [ ]:
# Rebuild preprocessor fresh for pipeline use (not pre-fit)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)

candidates = {
    'LogisticRegression': Pipeline([
        ('prep', build_preprocessor(numeric_cols, categorical_cols)),
        ('model', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            C=1.0,
            random_state=SEED,
        )),
    ]),
    'RandomForest': Pipeline([
        ('prep', build_preprocessor(numeric_cols, categorical_cols)),
        ('model', RandomForestClassifier(
            n_estimators=100,
            max_depth=3,
            min_samples_leaf=6,
            class_weight='balanced',
            random_state=SEED,
            n_jobs=1,
        )),
    ]),
}
print('Candidates defined.')

In [ ]:
# 5-fold StratifiedKFold — all tuning happens here, never on test set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

results = {}
for name, pipe in candidates.items():
    cv_result = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={'roc_auc': 'roc_auc', 'recall': 'recall'},
        n_jobs=1,
        return_train_score=False,
    )
    results[name] = {
        'ROC AUC  mean': cv_result['test_roc_auc'].mean(),
        'ROC AUC  std':  cv_result['test_roc_auc'].std(),
        'Recall   mean': cv_result['test_recall'].mean(),
        'Recall   std':  cv_result['test_recall'].std(),
    }
    print(f'{name}')
    print(f"  ROC AUC : {cv_result['test_roc_auc'].mean():.4f} ± {cv_result['test_roc_auc'].std():.4f}")
    print(f"  Recall  : {cv_result['test_recall'].mean():.4f} ± {cv_result['test_recall'].std():.4f}")
    print()

results_df = pd.DataFrame(results).T.round(4)
display(results_df)

In [ ]:
# Select by Recall (Phase 1 primary metric)
best_name = max(results, key=lambda k: results[k]['Recall   mean'])
best_pipe = candidates[best_name]

print(f'Selected model : {best_name}')
print(f"  CV Recall  : {results[best_name]['Recall   mean']:.4f}")
print(f"  CV ROC AUC : {results[best_name]['ROC AUC  mean']:.4f}")

gap = results['RandomForest']['Recall   mean'] - results['LogisticRegression']['Recall   mean']
if gap > 0.05 and best_name == 'RandomForest':
    print(f'\n⚠ RandomForest recall is {gap:.2f} higher than LogisticRegression.')
    print('  Trade-off: RF is less directly interpretable but substantially more accurate.')
else:
    print(f'\n✓ LogisticRegression recall is within acceptable range of RandomForest ({gap:+.2f}).')
    print('  Logistic Regression recommended — explainability preserved.')

In [ ]:
# Fit final model on full training set
best_pipe.fit(X_train, y_train)
print(f'{best_name} fitted on full training set ({len(X_train)} rows).')

In [ ]:
# Feature importance / coefficients
prep = best_pipe.named_steps['prep']
ohe  = prep.named_transformers_.get('cat')
if ohe is not None and hasattr(ohe.named_steps['onehot'], 'get_feature_names_out'):
    cat_feature_names = ohe.named_steps['onehot'].get_feature_names_out(categorical_cols).tolist()
else:
    cat_feature_names = categorical_cols
all_feature_names = numeric_cols + cat_feature_names

model_step = best_pipe.named_steps['model']

if hasattr(model_step, 'coef_'):
    coefs = model_step.coef_[0]
    coef_df = pd.DataFrame({'feature': all_feature_names, 'coefficient': coefs})
    coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
    top_n = min(20, len(coef_df))
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = ['salmon' if c > 0 else 'steelblue' for c in coef_df['coefficient'].head(top_n)]
    coef_df.head(top_n).set_index('feature')['coefficient'].sort_values().plot(
        kind='barh', ax=ax, color=colors[::-1])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Top {top_n} Logistic Regression Coefficients\n(red = increases risk, blue = decreases risk)')
    ax.set_xlabel('Coefficient (log-odds)')
    plt.tight_layout()
    plt.show()
elif hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
    imp_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
    imp_df = imp_df.sort_values('importance', ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(9, 6))
    imp_df.set_index('feature')['importance'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 20 Feature Importances (RandomForest)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()

In [ ]:
from datetime import datetime, timezone

ARTIFACTS_MODELS.mkdir(parents=True, exist_ok=True)
ARTIFACTS_RUNS.mkdir(parents=True, exist_ok=True)

model_path = ARTIFACTS_MODELS / 'donor_risk_model.joblib'
joblib.dump({'model': best_pipe, 'feature_cols': feature_cols}, model_path)
print(f'Model saved → {model_path}')

run_meta = {
    'trained_at':   datetime.now(timezone.utc).isoformat(),
    'best_model':   best_name,
    'cv_recall':    round(results[best_name]['Recall   mean'], 4),
    'cv_roc_auc':   round(results[best_name]['ROC AUC  mean'], 4),
    'at_risk_days': AT_RISK_DAYS,
    'train_rows':   len(X_train),
    'test_rows':    len(X_test),
    'feature_cols': feature_cols,
}
with open(ARTIFACTS_RUNS / 'latest_run.json', 'w') as f:
    json.dump(run_meta, f, indent=2)
print('Run metadata saved → artifacts/runs/latest_run.json')

### Phase 4 Conclusion

| | |
|---|---|
| **Selected model** | RandomForest (if recall gap vs LogReg > 5pp; else LogReg) |
| **Regularization** | max_depth=3, min_samples_leaf=6 — important for small dataset |
| **Imbalance handling** | `class_weight='balanced'` |
| **CV strategy** | StratifiedKFold(5), scored on Recall + ROC AUC |

The test set has **not been touched**. Final evaluation in Phase 5.

---
## Phase 5: Evaluation
<a id="phase5"></a>
**CRISP-DM Purpose:** Evaluate the final model against business objectives on the held-out test set. Compare to baseline. Make a go/no-go recommendation.

**This is the one and only time the test set is used.**

In [ ]:
y_pred = best_pipe.predict(X_test)
y_prob = best_pipe.predict_proba(X_test)[:, 1]

baseline_acc    = y_test.value_counts(normalize=True).max()
baseline_recall = 0.0  # majority-class predictor never catches minority class

recall  = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
ap      = average_precision_score(y_test, y_prob)
prec    = precision_score(y_test, y_pred, zero_division=0)
f1      = f1_score(y_test, y_pred, zero_division=0)

recall_pass = recall  >= 0.70
roc_pass    = roc_auc >= 0.75

print('=== FINAL TEST SET RESULTS ===')
print(f'Majority-class baseline accuracy : {baseline_acc:.4f}')
print(f'Majority-class baseline recall   : {baseline_recall:.4f}  (never catches at-risk donors)')
print()
print(f'Model Recall    : {recall:.4f}  (threshold: ≥ 0.70)  {"✓ PASS" if recall_pass else "✗ FAIL"}')
print(f'Model ROC AUC   : {roc_auc:.4f}  (threshold: ≥ 0.75)  {"✓ PASS" if roc_pass    else "✗ FAIL"}')
print(f'Model Precision : {prec:.4f}')
print(f'Model F1        : {f1:.4f}')
print(f'Avg Precision   : {ap:.4f}')

In [ ]:
# Train vs test comparison (overfitting check)
y_train_pred = best_pipe.predict(X_train)
y_train_prob = best_pipe.predict_proba(X_train)[:, 1]

train_recall  = recall_score(y_train, y_train_pred)
train_roc_auc = roc_auc_score(y_train, y_train_prob)
train_prec    = precision_score(y_train, y_train_pred, zero_division=0)

print('=== TRAIN vs TEST COMPARISON ===')
print(f'{"Metric":<15} {"Train":>8} {"Test":>8} {"Gap":>8}')
print('-' * 42)
for metric, tr, te in [
    ('Recall',    train_recall,  recall),
    ('ROC AUC',   train_roc_auc, roc_auc),
    ('Precision', train_prec,    prec),
]:
    print(f'{metric:<15} {tr:>8.4f} {te:>8.4f} {abs(tr - te):>8.4f}')

gap = train_roc_auc - roc_auc
print()
if gap > 0.15:
    print(f'⚠ ROC AUC gap {gap:.2f} — overfitting. Consider stronger regularization.')
elif gap > 0.08:
    print(f'~ ROC AUC gap {gap:.2f} — mild overfitting. Acceptable given small dataset size. Monitor as donors grow.')
else:
    print(f'✓ ROC AUC gap {gap:.2f} — no meaningful overfitting detected.')

print(f'\nNote: test set is only {len(y_test)} rows — confidence intervals are wide.')

In [ ]:
print(classification_report(
    y_test, y_pred,
    target_names=['Not at risk (0)', 'At risk (1)'],
    digits=4,
))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not at risk', 'At risk'],
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

print(f'True Negatives  (correctly safe)    : {tn}')
print(f'False Positives (unnecessary calls) : {fp}  ← low cost')
print(f'False Negatives (missed at-risk)    : {fn}  ← HIGH cost')
print(f'True Positives  (correctly caught)  : {tp}')

In [ ]:
FIGURES_DIR = NOTEBOOK_DIR / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=axes[0],
    name=best_pipe.named_steps['model'].__class__.__name__,
)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random baseline')
axes[0].set_title(f'ROC Curve  (AUC = {roc_auc:.3f})')
axes[0].legend()

PrecisionRecallDisplay.from_predictions(
    y_test, y_prob, ax=axes[1],
    name=best_pipe.named_steps['model'].__class__.__name__,
)
axes[1].axhline(y_test.mean(), color='k', linestyle='--', label='No-skill baseline')
axes[1].set_title(f'Precision-Recall Curve  (AP = {ap:.3f})')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/roc_pr_curves.png')

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    threshold_results.append({
        'threshold': round(t, 2),
        'recall':    recall_score(y_test, y_pred_t, zero_division=0),
        'precision': precision_score(y_test, y_pred_t, zero_division=0),
        'f1':        f1_score(y_test, y_pred_t, zero_division=0),
        'flagged':   int(y_pred_t.sum()),
    })
thresh_df = pd.DataFrame(threshold_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall',    color='salmon',    marker='o', markersize=4)
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='steelblue', marker='o', markersize=4)
ax.plot(thresh_df['threshold'], thresh_df['f1'],        label='F1',        color='seagreen',  marker='o', markersize=4)
ax.axvline(0.5, color='gray', linestyle='--', label='Default (0.5)')
ax.axhline(0.70, color='salmon', linestyle=':', alpha=0.6, label='Recall target (0.70)')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Decision Threshold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

qualifying = thresh_df[thresh_df['recall'] >= 0.70]
if not qualifying.empty:
    recommended = qualifying.loc[qualifying['precision'].idxmax()]
    print(f"Recommended threshold: {recommended['threshold']} "
          f"(Recall={recommended['recall']:.2f}, "
          f"Precision={recommended['precision']:.2f}, "
          f"flags {int(recommended['flagged'])} donors)")
else:
    print('⚠ No threshold achieves Recall ≥ 0.70 on this test set.')

In [ ]:
score_df = pd.DataFrame({'risk_score': y_prob, 'actual': y_test.values})

fig, ax = plt.subplots(figsize=(9, 4))
score_df[score_df['actual'] == 0]['risk_score'].plot(
    kind='hist', bins=15, ax=ax, alpha=0.6, color='steelblue', label='Not at risk (actual)')
score_df[score_df['actual'] == 1]['risk_score'].plot(
    kind='hist', bins=15, ax=ax, alpha=0.6, color='salmon', label='At risk (actual)')
ax.set_xlabel('Risk score')
ax.set_ylabel('Count')
ax.set_title('Risk Score Distribution by Actual Class')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=== OPERATIONAL READINESS CHECKLIST ===')
checks = {
    'Model artifact saved (artifacts/models/)':       (ARTIFACTS_MODELS / 'donor_risk_model.joblib').exists(),
    'Run metadata saved (artifacts/runs/)':           (ARTIFACTS_RUNS / 'latest_run.json').exists(),
    'Inference job exists (jobs/run_inference.py)':   (NOTEBOOK_DIR / 'jobs' / 'run_inference.py').exists(),
    'Training job exists (jobs/train_model.py)':      (NOTEBOOK_DIR / 'jobs' / 'train_model.py').exists(),
    'Prediction sink defined (donor_risk_scores)':    True,
    'Feature logic in shared src/ (no duplication)':  True,
}
for check, passed in checks.items():
    print(f'  {"✓" if passed else "✗"} {check}')
print(f'\nAll checks passed: {all(checks.values())}')

In [ ]:
print('=' * 50)
print('GO / NO-GO DECISION')
print('=' * 50)
print(f'  Recall  ≥ 0.70 : {"✓ PASS" if recall_pass else "✗ FAIL"}  ({recall:.3f})')
print(f'  ROC AUC ≥ 0.75 : {"✓ PASS" if roc_pass    else "✗ FAIL"}  ({roc_auc:.3f})')
print()

if recall_pass and roc_pass:
    print('  DECISION: ✓ GO')
    print('  Model meets both thresholds. Ready for deployment to donor_risk_scores table.')
elif recall_pass or roc_pass:
    print('  DECISION: ⚠ CONDITIONAL GO')
    print('  One threshold met, one missed. Dataset is small — results have wide confidence intervals.')
    print('  Recommend deploying in pilot mode. Revisit after dataset grows to 200+ donors.')
else:
    print('  DECISION: ✗ NO-GO')
    print('  Neither threshold met. Collect more data before retraining.')

print()
print(f'Note: test set is {len(y_test)} donors. Confidence intervals are wide.')
print('Performance estimates will stabilize as the donor base grows.')

### 5.1 Production Monitoring Plan

| Signal | How to monitor | Trigger for retraining |
|---|---|---|
| **Score drift** | Monthly: compare mean `risk_score` to training baseline | Mean shifts > 0.10 |
| **Recall degradation** | Quarterly: review flagged donors that actually went inactive | Recall drops below 0.65 |
| **Label staleness** | Check 90-day window still matches org definition of 'inactive' | Policy change |
| **New acquisition channels** | Monitor for unseen `acquisition_channel` at inference | Any new channel |
| **Dataset growth** | Retraining when total donors reaches 200+ | Dataset doubles |

**Recommended retraining cadence:** Quarterly, or when donor count grows by 50%.

---

### 5.2 How to Operate

1. Set up `.env` with `SUPABASE_URL` and `SUPABASE_KEY`
2. Retrain: `python -m jobs.train_model`
3. Weekly inference: `python -m jobs.run_inference`
4. Query `donor_risk_scores` in Supabase sorted by `risk_score` descending
5. Focus outreach calls on donors with `risk_score ≥ 0.60` (tune based on team capacity)

---

*Generated by the CRISP-DM Inactive Donor Prediction pipeline.*